# 🧪 Thực Nghiệm Nâng Cao RS-LiDAR: Guidance Scale Stress-Test & Particle Scaling (Kaggle 2x T4)
### **Khung Thực Nghiệm Độc Lập**: Kiểm chứng Động Học Manifold & Đường Biên Pareto Tối Ưu

Notebook này hiện thực hóa **2 Bản Đề Xuất Thực Nghiệm Khoa Học** nhằm chứng minh các ưu thế vượt trội của **RS-LiDAR** so với **LiDAR gốc (ICML 2026)**:
1. **Thực Nghiệm 1 (Guidance Scale Stress-Test: $s \in \{7.5, 12.5, 15.0, 17.5, 20.0\}$)**: 
   - Kiểm chứng giả thuyết: LiDAR gốc bị rung giật gradient (chứng minh từ Test 3), khi ép scale $s \ge 17.5$ lực giật bị phóng đại làm vỡ đa tạp, ảnh bị hỏng và điểm sụt giảm.
   - RS-LiDAR nhờ chặn Lipschitz chặt chẽ ($L_\sigma \le \frac{\lambda}{\sigma \sqrt{2\pi}}$) có **Ngưỡng Chịu Lực (Stability Margin)** rộng hơn hẳn, duy trì ảnh sắc nét và đạt đỉnh ImageReward mới ở $s = 17.5 \sim 20.0$.
2. **Thực Nghiệm 2 (Particle Scaling: $N \in \{9, 20, 50, 100\}$)**:
   - Kiểm chứng giả thuyết: LiDAR gốc bị bão hòa (Best-of-1 Trap từ Test 2), tăng từ 50 lên 100 hạt không tăng thêm chất lượng.
   - RS-LiDAR kích hoạt sức mạnh tập hợp đa hạt (Multi-particle Consensus), tiếp tục bứt phá tại $N=100$, mở rộng đường biên Pareto Frontier.

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Runtime Settings.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi


## 2. Cấu Hình Tham Số Thực Nghiệm Tập Trung
Bạn có thể chọn chạy **Thực Nghiệm 1 (Guidance Scale Sweep)** hoặc **Thực Nghiệm 2 (Particle Scaling)**. Chế độ mặc định là `1_GUIDANCE_SCALE_SWEEP` vì có thể tái sử dụng ngay $N=50$ hạt Phase 1 đã chạy từ trước, hoàn tất chỉ trong ~15–20 phút!

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH TẬP TRUNG (CHỈ CẦN ĐIỀU CHỈNH TẠI ĐÂY)
# ==============================================================================
# Chọn chế độ thực nghiệm: 
#   '1_GUIDANCE_SCALE_SWEEP' : Chạy quét Scale s ∈ [7.5, 12.5, 15.0, 17.5, 20.0] (Rất nhanh ~15 phút nếu tái sử dụng Phase 1)
#   '2_PARTICLE_SCALING'     : Chạy quét số hạt N ∈ [9, 20, 50, 100]
#   'BOTH'                   : Chạy cả hai thực nghiệm
EXPERIMENT_MODE = '1_GUIDANCE_SCALE_SWEEP'

NUM_PROMPTS = 20                  # Số prompt thực nghiệm (mặc định 20 prompts từ GenEval)
REUSE_EXISTING_PHASE1 = True      # Tái sử dụng Phase 1 nếu đã có trong Lookahead_samples (tiết kiệm 100% thời gian Phase 1)
LOOKAHEAD_STEPS = 5               # Số bước DPM lookahead Phase 1
LOOKAHEAD_SEED = 100              # Seed sinh hạt lookahead

# Cấu hình dải tham số
GUIDANCE_SCALES_SWEEP = [7.5, 12.5, 15.0, 17.5, 20.0]  # Dải Scale cho Exp 1
PARTICLE_COUNTS_SWEEP = [9, 20, 50, 100]               # Dải hạt cho Exp 2

# Tham số RS-LiDAR chuẩn
SIGMA = 1.0                       # Độ lệch chuẩn Randomized Smoothing cho RS-LiDAR
NUM_MC_SAMPLES = 4                # Số mẫu Monte Carlo tính E[R(x+xi)]
LAMBDA = 5000                     # Hệ số Softmax reward chuẩn bài báo
# ==============================================================================


## 3. Đồng Bộ Repository & Cài Đặt Thư Viện Cần Thiết
Tự động clone code mới nhất từ GitHub và cài đặt môi trường tối giản, không xung đột dependency.

In [ ]:
import os, shutil, glob, json, random, subprocess, threading, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# 1. Đồng bộ mã nguồn
REPO_DIR = '/kaggle/working/RS-LiDAR'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài đặt các thư viện phụ thuộc chuẩn
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q scipy matplotlib seaborn pandas tabulate open_clip_torch

# 3. Hàm hỗ trợ chạy song song GPU
def run_parallel_or_serial(cmd0, cmd1=None):
    n_g = torch.cuda.device_count() if torch.cuda.is_available() else 1
    if n_g >= 2 and cmd1 is not None:
        p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def stream(p, pfx):
            for line in iter(p.stdout.readline, ''):
                if line.strip(): print(f"{pfx} {line.strip()}")
            p.stdout.close()
        t0 = threading.Thread(target=stream, args=(p0, '[GPU 0]')); t1 = threading.Thread(target=stream, args=(p1, '[GPU 1]'))
        t0.start(); t1.start(); t0.join(); t1.join()
        return (p0.wait() == 0 and p1.wait() == 0)
    else:
        res = subprocess.run(cmd0, shell=True)
        return (res.returncode == 0)

print("✅ Môi trường đã sẵn sàng!")


## 4. Chuẩn Bị Tập Prompt Thực Nghiệm (Tái Lập Đảm Bảo 100% Seed 42)

In [ ]:
PROMPT_FILE = 'prompt_files/geneval_metadata.jsonl'
EXP_PROMPT_FILE = '/kaggle/working/exp_prompts_{NUM_PROMPTS}.jsonl'

all_prompts = []
with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): all_prompts.append(json.loads(line.strip()))

random.seed(42)
selected_prompts = random.sample(all_prompts, min(NUM_PROMPTS, len(all_prompts)))

with open(EXP_PROMPT_FILE, 'w', encoding='utf-8') as f:
    for item in selected_prompts:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"✅ Đã chọn và lưu {len(selected_prompts)} prompts chuẩn vào: {EXP_PROMPT_FILE}")


## 5. [PHASE 1] Lookahead Sampling & Tái Sử Dụng Latent Sẵn Có
Nếu bạn đã chạy Phase 1 trước đó, cell này tự động nhận diện và tái sử dụng 100%, không tốn thời gian sinh lại.

In [ ]:
LOOKAHEAD_DIR = '/kaggle/working/Lookahead_samples' if is_kaggle else '/content/Lookahead_samples'
os.makedirs(LOOKAHEAD_DIR, exist_ok=True)

n_existing = len(glob.glob(f"{LOOKAHEAD_DIR}/*/results.json"))
print(f"🔍 Kiểm tra Phase 1: Đang có sẵn {n_existing}/{NUM_PROMPTS} prompts tại {LOOKAHEAD_DIR}")

if REUSE_EXISTING_PHASE1 and n_existing >= NUM_PROMPTS:
    print("⏩ TÁI SỬ DỤNG 100%: Toàn bộ Phase 1 lookahead samples đã có sẵn! Bỏ qua Phase 1 để tiết kiệm tính toán.")
else:
    print(f"🚀 Đang sinh Phase 1 Lookahead Samples (N=50 hạt, DPM-5)...\n")
    n_g = torch.cuda.device_count() if torch.cuda.is_available() else 1
    if n_g >= 2:
        cmd0 = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles 50 --num_inference_steps {LOOKAHEAD_STEPS} --seed {LOOKAHEAD_SEED} --save_individual_images True --output_dir '{LOOKAHEAD_DIR}' --num_shards 2 --shard_id 0 --gpu_id 0 --resume"
        cmd1 = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles 50 --num_inference_steps {LOOKAHEAD_STEPS} --seed {LOOKAHEAD_SEED} --save_individual_images True --output_dir '{LOOKAHEAD_DIR}' --num_shards 2 --shard_id 1 --gpu_id 1 --resume"
        run_parallel_or_serial(cmd0, cmd1)
    else:
        cmd = f"python -u lookahead_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --model_name 'runwayml/stable-diffusion-v1-5' --num_particles 50 --num_inference_steps {LOOKAHEAD_STEPS} --seed {LOOKAHEAD_SEED} --save_individual_images True --output_dir '{LOOKAHEAD_DIR}' --resume"
        run_parallel_or_serial(cmd)
    print("✅ Hoàn tất sinh Phase 1 Lookahead!")


## 6. [PHASE 2] Thực Thi Target Sampling Cho Toàn Bộ Dải Khảo Sát
Tự động chạy lần lượt các cấu hình của Thực nghiệm (phân bổ song song 2 GPU trên Kaggle nếu có), ghi nhận kết quả và lưu vào thư mục `Target_samples/`.

In [ ]:
TARGET_OUTPUT_ROOT = '/kaggle/working/Target_samples' if is_kaggle else '/content/Target_samples'
os.makedirs(TARGET_OUTPUT_ROOT, exist_ok=True)
n_g = torch.cuda.device_count() if torch.cuda.is_available() else 1

runs_to_execute = []

if EXPERIMENT_MODE in ['1_GUIDANCE_SCALE_SWEEP', 'BOTH']:
    print(f"📋 Chuẩn bị chạy Thực Nghiệm 1: Guidance Scale Sweep với s ∈ {GUIDANCE_SCALES_SWEEP}...")
    for s_val in GUIDANCE_SCALES_SWEEP:
        # 1. LiDAR Gốc (sigma = 0)
        runs_to_execute.append({
            'exp': 'Exp1_Scale',
            'method': 'Vanilla_LiDAR',
            'scale': s_val,
            'n_particles': 50,
            'sigma': 0.0,
            'run_name': f"LiDAR_s{s_val}_n50_lmbda{LAMBDA}"
        })
        # 2. RS-LiDAR (sigma = 1.0, M = 4)
        runs_to_execute.append({
            'exp': 'Exp1_Scale',
            'method': 'RS_LiDAR',
            'scale': s_val,
            'n_particles': 50,
            'sigma': SIGMA,
            'run_name': f"RSLiDAR_s{s_val}_n50_sig{SIGMA}_M{NUM_MC_SAMPLES}_lmbda{LAMBDA}"
        })

if EXPERIMENT_MODE in ['2_PARTICLE_SCALING', 'BOTH']:
    print(f"📋 Chuẩn bị chạy Thực Nghiệm 2: Particle Scaling với N ∈ {PARTICLE_COUNTS_SWEEP}...")
    for n_val in PARTICLE_COUNTS_SWEEP:
        runs_to_execute.append({
            'exp': 'Exp2_Particle',
            'method': 'Vanilla_LiDAR',
            'scale': 12.5,
            'n_particles': n_val,
            'sigma': 0.0,
            'run_name': f"LiDAR_s12.5_n{n_val}_lmbda{LAMBDA}"
        })
        runs_to_execute.append({
            'exp': 'Exp2_Particle',
            'method': 'RS_LiDAR',
            'scale': 12.5,
            'n_particles': n_val,
            'sigma': SIGMA,
            'run_name': f"RSLiDAR_s12.5_n{n_val}_sig{SIGMA}_M{NUM_MC_SAMPLES}_lmbda{LAMBDA}"
        })

print(f"🚀 TỔNG SỐ LƯỢT CHẠY TARGET SAMPLING: {len(runs_to_execute)} runs")

for idx_run, run_cfg in enumerate(runs_to_execute):
    r_name = run_cfg['run_name']
    s_val = run_cfg['scale']
    n_parts = run_cfg['n_particles']
    print("\n" + "="*90)
    print(f"[{idx_run+1}/{len(runs_to_execute)}] ĐANG CHẠY: {r_name} (Scale={s_val}, N={n_parts})")
    print("="*90)
    
    if n_g >= 2:
        cmd0 = f"python -u LiDAR_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --lookahead_path '{LOOKAHEAD_DIR}' --scale {s_val} --top_k {n_parts} --lmbda {LAMBDA} --num_inference_steps 50 --eta 0.0 --use_rag --resample_t_end 200 --run_name '{r_name}' --num_shards 2 --shard_id 0 --gpu_id 0 --resume"
        cmd1 = f"python -u LiDAR_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --lookahead_path '{LOOKAHEAD_DIR}' --scale {s_val} --top_k {n_parts} --lmbda {LAMBDA} --num_inference_steps 50 --eta 0.0 --use_rag --resample_t_end 200 --run_name '{r_name}' --num_shards 2 --shard_id 1 --gpu_id 1 --resume"
        run_parallel_or_serial(cmd0, cmd1)
    else:
        cmd = f"python -u LiDAR_sampling.py --prompt_path '{EXP_PROMPT_FILE}' --lookahead_path '{LOOKAHEAD_DIR}' --scale {s_val} --top_k {n_parts} --lmbda {LAMBDA} --num_inference_steps 50 --eta 0.0 --use_rag --resample_t_end 200 --run_name '{r_name}' --resume"
        run_parallel_or_serial(cmd)

print("\n🎉 HOÀN TẤT TOÀN BỘ CÁC LƯỢT CHẠY TARGET SAMPLING!")


## 7. Báo Cáo Kết Quả, Bảng Tổng Hợp & Đồ Thị Động Học Khoa Học
Tự động đọc toàn bộ kết quả, in bảng so sánh, vẽ đồ thị phản ánh ranh giới sụp đổ của LiDAR vs sức bền của RS-LiDAR.

In [ ]:
import glob, json, os, numpy as np, pandas as pd, matplotlib.pyplot as plt

# Tổng hợp kết quả từ các run
summary_rows = []
target_runs = glob.glob(f"{TARGET_OUTPUT_ROOT}/*")

for r_dir in sorted(target_runs):
    r_name = os.path.basename(r_dir)
    metrics_f = os.path.join(r_dir, 'final_metrics.json')
    ir_val, clip_val, hps_val = np.nan, np.nan, np.nan
    
    # Đọc từ final_metrics.json nếu có
    if os.path.exists(metrics_f):
        try:
            with open(metrics_f, 'r') as f:
                m_data = json.load(f)
                ir_val = m_data.get('ImageReward', {}).get('mean', np.nan)
                clip_val = m_data.get('Clip-Score', {}).get('mean', np.nan)
                hps_val = m_data.get('HumanPreference', {}).get('mean', np.nan)
        except Exception:
            pass
    
    # Nếu chưa có file tổng, đọc trung bình từ results.json của từng prompt
    if np.isnan(ir_val):
        p_results = glob.glob(f"{r_dir}/*/results.json")
        ir_list, clip_list = [], []
        for pr_f in p_results:
            try:
                with open(pr_f, 'r') as f:
                    data = json.load(f)
                    if 'ImageReward' in data: ir_list.append(data['ImageReward']['mean'])
                    if 'Clip-Score' in data: clip_list.append(data['Clip-Score']['mean'])
            except Exception:
                pass
        if ir_list: ir_val = float(np.mean(ir_list))
        if clip_list: clip_val = float(np.mean(clip_list))
        
    # Phân loại
    method_type = 'RS_LiDAR' if 'RSLiDAR' in r_name else 'Vanilla_LiDAR'
    scale_val = 12.5
    for s_v in [7.5, 12.5, 15.0, 17.5, 20.0]:
        if f"_s{s_v}_" in r_name:
            scale_val = s_v; break
    
    n_part_val = 50
    for n_v in [9, 20, 50, 100]:
        if f"_n{n_v}_" in r_name:
            n_part_val = n_v; break
            
    summary_rows.append({
        'Run_Name': r_name,
        'Method': method_type,
        'Guidance_Scale': scale_val,
        'Particles_N': n_part_val,
        'ImageReward': round(ir_val, 4) if not np.isnan(ir_val) else 'N/A',
        'CLIP_Score': round(clip_val, 4) if not np.isnan(clip_val) else 'N/A',
        'HPS_v2.1': round(hps_val, 4) if not np.isnan(hps_val) else 'N/A'
    })

df_summary = pd.DataFrame(summary_rows)
print("="*105)
print("📊 BẢNG TỔNG HỢP KẾT QUẢ THỰC NGHIỆM ĐỀ XUẤT:")
print("="*105)
from IPython.display import display
display(df_summary)

# Lưu CSV
csv_out_path = '/kaggle/working/advanced_experiments_summary.csv' if is_kaggle else '/content/advanced_experiments_summary.csv'
df_summary.to_csv(csv_out_path, index=False)
print(f"💾 Đã lưu bảng tổng hợp tại: {csv_out_path}")

# Vẽ đồ thị cho Thực Nghiệm 1 nếu có dải scale
df_exp1 = df_summary[df_summary['Particles_N'] == 50].copy()
if len(df_exp1) > 2:
    try:
        plt.figure(figsize=(10, 6))
        for m_name, color, marker, ls in [('Vanilla_LiDAR', 'red', 'o', '--'), ('RS_LiDAR', 'green', 's', '-')]:
            sub = df_exp1[df_exp1['Method'] == m_name].sort_values('Guidance_Scale')
            if len(sub) > 0 and 'ImageReward' in sub.columns:
                valid = sub[sub['ImageReward'] != 'N/A']
                plt.plot(valid['Guidance_Scale'], valid['ImageReward'].astype(float), label=f"{m_name}", color=color, marker=marker, linestyle=ls, linewidth=2.5)
        plt.title("Experiment 1: Guidance Scale Stress-Test (Lipschitz Stability Margin)", fontsize=13, fontweight='bold')
        plt.xlabel("Guidance Scale s", fontsize=12)
        plt.ylabel("ImageReward ↑", fontsize=12)
        plt.axvline(x=15.0, color='gray', linestyle=':', label='LiDAR Peak Threshold (s=15.0)')
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(fontsize=11)
        plt.tight_layout()
        p1_plot_f = '/kaggle/working/exp1_guidance_scale_curve.png' if is_kaggle else '/content/exp1_guidance_scale_curve.png'
        plt.savefig(p1_plot_f, dpi=200)
        plt.show()
        print(f"📈 Đã lưu đồ thị Thực Nghiệm 1 tại: {p1_plot_f}")
    except Exception as e:
        print(f"Vẽ đồ thị Exp 1 lỗi: {e}")


## 8. Đóng Gói Toàn Bộ Kết Quả & Ảnh Độc Quyền (1-Click Download)
Nén toàn bộ bảng CSV, đồ thị biểu diễn và các ảnh đã sinh ra thành file zip để tải về chỉ với 1 cú click.

In [ ]:
zip_path = '/kaggle/working/rs_lidar_advanced_results.zip'
print(f"📦 Đang nén toàn bộ kết quả và ảnh thực nghiệm vào: {zip_path}...")
!zip -r -q {zip_path} {TARGET_OUTPUT_ROOT} /kaggle/working/advanced_experiments_summary.csv
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\n✅ HOÀN TẤT ĐÓNG GÓI! Kích thước file: {size_mb:.2f} MB")
    print("Tải file zip trực tiếp từ tab Output bên phải Kaggle để xem toàn bộ ảnh và bảng số liệu!")
